In [ ]:
import os
import pandas as pd
import kagglehub

# 1. Download latest version (Your starting point)
path = kagglehub.dataset_download("amananandrai/ag-news-classification-dataset")
print("Path to dataset files:", path)

# 2. List all files in the downloaded directory to find the exact CSV names
print("\n--- Files in Dataset ---")
files = os.listdir(path)
print(files)

# 3. Load the dataset 
# (Assuming the main file is named 'train.csv' or similar based on typical AG News datasets. 
# Update 'train.csv' to the actual filename printed in the step above if it differs.)
file_path = os.path.join(path, 'train.csv')
df = pd.read_csv(file_path)

# 4. Check max rows and columns
print("\n--- Dataset Shape ---")
print(f"Total Rows (Max Rows): {df.shape[0]}")
print(f"Total Columns: {df.shape[1]}")

# 5. Check column names and data types
print("\n--- Data Types & Column Info ---")
# df.info() provides a concise summary including column names, non-null counts, and data types
df.info() 


print("\n--- Some Random Samples ---")
display(df.sample(10))

> Data preparation (B)

In [ ]:
import os
import re
import json
import pandas as pd
import kagglehub

# Download dataset
path = kagglehub.dataset_download("amananandrai/ag-news-classification-dataset")

# Load both splits
train_raw = pd.read_csv(os.path.join(path, "train.csv"))
test_raw = pd.read_csv(os.path.join(path, "test.csv"))

print(f"Raw train: {train_raw.shape}, Raw test: {test_raw.shape}")

# --- CONFIG ---
SAMPLES_PER_CLASS_TRAIN = 12500  # 12500 x 4 classes = 50,000 total
SAMPLES_PER_CLASS_TEST = 1900    # 1900 x 4 = 7,600 for test
RANDOM_SEED = 42


# --- CLEANING ---
def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.strip()
    text = re.sub(r"#39;", "'", text)
    text = re.sub(r"&amp;", "&", text)
    text = re.sub(r'quot;', '"', text)
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def process_split(df, samples_per_class=None):
    df = df.copy()

    # Combine title + description
    df["text"] = df["Title"].apply(clean_text) + " " + df["Description"].apply(clean_text)

    # Shift labels: 1-4 → 0-3 (HuggingFace expects 0-indexed)
    df["label"] = df["Class Index"] - 1

    # Drop bad rows
    df = df.dropna(subset=["text", "label"])
    df = df[df["text"].str.len() > 0]
    df = df.drop_duplicates(subset=["text"])

    # Subsample
    if samples_per_class:
        df = df.groupby("label").apply(
            lambda x: x.sample(n=min(samples_per_class, len(x)), random_state=RANDOM_SEED)
        ).reset_index(drop=True)

    return df[["text", "label"]]


# Process
train_df = process_split(train_raw, samples_per_class=SAMPLES_PER_CLASS_TRAIN)
test_df = process_split(test_raw, samples_per_class=SAMPLES_PER_CLASS_TEST)

print(f"\nProcessed train: {train_df.shape}")
print(f"Processed test: {test_df.shape}")
print(f"\nTrain label distribution:\n{train_df['label'].value_counts().sort_index()}")
print(f"\nTest label distribution:\n{test_df['label'].value_counts().sort_index()}")
print(f"\nSample rows:")
display(train_df.sample(5))

> Saving files for github

In [ ]:
id2label = {
    "0": "World",
    "1": "Sports",
    "2": "Business",
    "3": "Sci/Tech"
}

with open("id2label.json", "w") as f:
    json.dump(id2label, f, indent=2)

print("Saved id2label.json")
print(json.dumps(id2label, indent=2))

In [ ]:
# Save data_prep.py as a standalone script file

data_prep_code = '''import os
import re
import json
import pandas as pd
import kagglehub

SAMPLES_PER_CLASS_TRAIN = 12500
SAMPLES_PER_CLASS_TEST = 1900
RANDOM_SEED = 42


def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.strip()
    text = re.sub(r"#39;", "'", text)
    text = re.sub(r"&amp;", "&", text)
    text = re.sub(r'quot;', '"', text)
    text = re.sub(r"http\\S+|www\\S+", "", text)
    text = re.sub(r"\\s+", " ", text)
    return text.strip()


def process_split(df, samples_per_class=None):
    df = df.copy()
    df["text"] = df["Title"].apply(clean_text) + " " + df["Description"].apply(clean_text)
    df["label"] = df["Class Index"] - 1
    df = df.dropna(subset=["text", "label"])
    df = df[df["text"].str.len() > 0]
    df = df.drop_duplicates(subset=["text"])
    if samples_per_class:
        df = df.groupby("label").apply(
            lambda x: x.sample(n=min(samples_per_class, len(x)), random_state=RANDOM_SEED)
        ).reset_index(drop=True)
    return df[["text", "label"]]


def main():
    path = kagglehub.dataset_download("amananandrai/ag-news-classification-dataset")
    train_raw = pd.read_csv(os.path.join(path, "train.csv"))
    test_raw = pd.read_csv(os.path.join(path, "test.csv"))

    train_df = process_split(train_raw, SAMPLES_PER_CLASS_TRAIN)
    test_df = process_split(test_raw, SAMPLES_PER_CLASS_TEST)

    id2label = {"0": "World", "1": "Sports", "2": "Business", "3": "Sci/Tech"}
    with open("id2label.json", "w") as f:
        json.dump(id2label, f, indent=2)

    os.makedirs("data", exist_ok=True)
    train_df.to_csv("data/train.csv", index=False)
    test_df.to_csv("data/test.csv", index=False)
    print(f"Train: {len(train_df)}, Test: {len(test_df)}")


if __name__ == "__main__":
    main()
'''

os.makedirs("src", exist_ok=True)
with open("src/data_prep.py", "w") as f:
    f.write(data_prep_code)

print("✅ Saved src/data_prep.py")

In [ ]:
os.makedirs("data", exist_ok=True)
train_df.to_csv("data/train.csv", index=False)
test_df.to_csv("data/test.csv", index=False)
print("Saved data/train.csv and data/test.csv")